# NanoDet-Plus on KITTI (separate notebook)
NanoDet's official code needs an older stack (`pytorch-lightning` 1.x), so it runs here, in its own Colab session, using the **same KITTI split and the same evaluator and latency method** as the main notebook. It saves `result_nanodet.json` into the same `adas_benchmark/results` folder.

**Honest note:** this notebook is best-effort. NanoDet's repo is old and Colab's libraries change, so a cell may need a small fix. If it cannot be made to work in time, present NanoDet as future work and compare the other three models.

Run with a T4 GPU runtime. Set `QUICK = True` first for a 15-minute test.

## 1. Setup

In [ ]:
!pip -q install ultralytics torchmetrics pycocotools omegaconf imagesize pyaml termcolor tabulate tensorboard "pytorch-lightning>=1.9,<2.0"
!git clone -q --depth 1 https://github.com/RangiLyu/nanodet.git /content/nanodet
!wget -q -O /content/nanodet-plus-m_320_checkpoint.ckpt https://github.com/RangiLyu/nanodet/releases/download/v1.0.0-alpha-1/nanodet-plus-m_320_checkpoint.ckpt
import sys, torch
sys.path.insert(0, "/content/nanodet")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU found")

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/adas_benchmark")
except Exception as e:
    print("Drive not mounted, saving locally:", e)
    OUT = Path("/content/adas_benchmark")
RES, WTS, FIG = OUT / "results", OUT / "weights", OUT / "figures"
for p in (RES, WTS, FIG):
    p.mkdir(parents=True, exist_ok=True)

QUICK = False
DATA = "kitti.yaml"
INPUT_SIZE = 416            # NanoDet input (the 320 checkpoint is fully convolutional, so any size works)
EPOCHS = 1 if QUICK else 40
BATCH = 32
SEED = 0
MAX_TRAIN, MAX_VAL = (200, 100) if QUICK else (None, None)
import os
CPU_THREADS = min(4, os.cpu_count() or 1)

In [ ]:
import os, io, time, json, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, cv2, torch
from tqdm.auto import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
VEHICLE_NAMES = {"car", "van", "truck", "tram", "bus"}   # classes counted as "vehicles"

def list_images(p):
    if isinstance(p, (list, tuple)):
        p = p[0]
    return sorted(str(x) for x in Path(p).rglob("*") if x.suffix.lower() in IMG_EXT)

def label_path(img_path):
    a, b = str(img_path).rsplit(os.sep + "images" + os.sep, 1)
    return Path(a, "labels", b).with_suffix(".txt")

def read_targets(img_path, w, h):
    """YOLO txt labels -> boxes in pixel xyxy + class ids."""
    boxes, labels = [], []
    lp = label_path(img_path)
    if lp.exists():
        for line in lp.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            c = int(float(parts[0]))
            cx, cy, bw, bh = [float(v) for v in parts[1:5]]
            boxes.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
            labels.append(c)
    return np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=np.int64)

def yolo_to_coco(img_list, names, root, out_json):
    images, anns, aid = [], [], 1
    for i, p in enumerate(img_list, 1):
        im = cv2.imread(p)
        h, w = im.shape[:2]
        images.append({"id": i, "file_name": os.path.relpath(p, root), "width": w, "height": h})
        b, l = read_targets(p, w, h)
        for (x1, y1, x2, y2), c in zip(b, l):
            bw, bh = float(x2 - x1), float(y2 - y1)
            if bw < 1 or bh < 1:
                continue
            anns.append({"id": aid, "image_id": i, "category_id": int(c) + 1,
                         "bbox": [float(x1), float(y1), bw, bh], "area": bw * bh, "iscrowd": 0})
            aid += 1
    cats = [{"id": k + 1, "name": v} for k, v in sorted(names.items())]
    with open(out_json, "w") as f:
        json.dump({"images": images, "annotations": anns, "categories": cats}, f)

def _t(a, dtype):
    return torch.as_tensor(np.asarray(a, dtype=dtype))

def evaluate_model(predict_fn, img_list, names):
    """ONE evaluator for every model: same images, same torchmetrics mAP.
    predict_fn(bgr_image) -> (boxes_xyxy, scores, class_ids)."""
    vid = torch.tensor([i for i, n in names.items() if n in VEHICLE_NAMES], dtype=torch.int64)
    m_all = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    m_veh = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    for p in tqdm(img_list, desc="evaluating"):
        img = cv2.imread(p)
        h, w = img.shape[:2]
        gb, gl = read_targets(p, w, h)
        pb, ps, pl = predict_fn(img)
        tgt = {"boxes": _t(gb, np.float32).reshape(-1, 4), "labels": _t(gl, np.int64)}
        prd = {"boxes": _t(pb, np.float32).reshape(-1, 4), "scores": _t(ps, np.float32), "labels": _t(pl, np.int64)}
        m_all.update([prd], [tgt])
        gk, pk = torch.isin(tgt["labels"], vid), torch.isin(prd["labels"], vid)
        m_veh.update([{k: v[pk] for k, v in prd.items()}], [{k: v[gk] for k, v in tgt.items()}])
    ra, rv = m_all.compute(), m_veh.compute()
    return {"map50": float(ra["map_50"]), "map": float(ra["map"]),
            "map50_vehicle": float(rv["map_50"]), "map_vehicle": float(rv["map"])}

def measure_latency(predict_fn, img_list, cuda, n=100, warmup=10):
    """End-to-end latency per image (pre-processing + model + post-processing), batch size 1."""
    pick = random.Random(0).sample(img_list, min(n + warmup, len(img_list)))
    imgs = [cv2.imread(p) for p in pick]
    for im in imgs[:warmup]:
        predict_fn(im)
    lat = []
    for im in imgs[warmup:]:
        if cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        predict_fn(im)
        if cuda:
            torch.cuda.synchronize()
        lat.append((time.perf_counter() - t0) * 1000)
    lat = np.array(lat)
    return {"mean_ms": float(lat.mean()), "p95_ms": float(np.percentile(lat, 95)), "fps": float(1000 / lat.mean())}

def save_result(res_dir, tag, d):
    Path(res_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(res_dir) / f"result_{tag}.json", "w") as f:
        json.dump(d, f, indent=2)
    return d

## 2. Same KITTI split, converted to COCO format for NanoDet

In [ ]:
from ultralytics.data.utils import check_det_dataset
d = check_det_dataset(DATA)
names = {int(k): v for k, v in d["names"].items()} if isinstance(d["names"], dict) else dict(enumerate(d["names"]))
NC = len(names)
train_imgs, val_imgs = list_images(d["train"]), list_images(d["val"])
random.Random(SEED).shuffle(train_imgs)          # identical shuffle/limits as the main notebook
if MAX_TRAIN:
    train_imgs, val_imgs = train_imgs[:MAX_TRAIN], val_imgs[:MAX_VAL]
train_root = str(Path(d["train"] if isinstance(d["train"], str) else d["train"][0]))
val_root = str(Path(d["val"] if isinstance(d["val"], str) else d["val"][0]))
Path("/content/coco_kitti").mkdir(exist_ok=True)
yolo_to_coco(train_imgs, names, train_root, "/content/coco_kitti/train.json")
yolo_to_coco(val_imgs, names, val_root, "/content/coco_kitti/val.json")
print("classes:", names, "| train:", len(train_imgs), "| val:", len(val_imgs))

## 3. Config and training

In [ ]:
import yaml
cfg_d = yaml.safe_load(open("/content/nanodet/config/nanodet-plus-m_320.yml"))
cfg_d["save_dir"] = "/content/nanodet_workspace"
cfg_d["model"]["arch"]["head"]["num_classes"] = NC
cfg_d["model"]["arch"]["aux_head"]["num_classes"] = NC
for split, root, ann in (("train", train_root, "/content/coco_kitti/train.json"), ("val", val_root, "/content/coco_kitti/val.json")):
    cfg_d["data"][split]["img_path"] = root
    cfg_d["data"][split]["ann_path"] = ann
    cfg_d["data"][split]["input_size"] = [INPUT_SIZE, INPUT_SIZE]
cfg_d["device"].update({"gpu_ids": [0] if torch.cuda.is_available() else -1, "workers_per_gpu": 2, "batchsize_per_gpu": BATCH})
cfg_d["schedule"]["load_model"] = "/content/nanodet-plus-m_320_checkpoint.ckpt"
cfg_d["schedule"]["total_epochs"] = EPOCHS
cfg_d["schedule"]["lr_schedule"]["T_max"] = EPOCHS
cfg_d["schedule"]["warmup"]["steps"] = 200
cfg_d["schedule"]["val_intervals"] = 1 if QUICK else 5
cfg_d["class_names"] = [names[i] for i in range(NC)]
yaml.safe_dump(cfg_d, open("/content/nanodet_kitti.yml", "w"), sort_keys=False)
print("config written")

In [ ]:
# Training can take 1+ hours. Progress and validation mAP print as it goes.
!cd /content/nanodet && PYTHONPATH=/content/nanodet TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python tools/train.py /content/nanodet_kitti.yml --seed 0

## 4. Evaluate with the common evaluator, measure speed and size

In [ ]:
import io, contextlib
from nanodet.data.batch_process import stack_batch_img
from nanodet.data.collate import naive_collate
from nanodet.data.transform import Pipeline
from nanodet.model.arch import build_model
from nanodet.util import Logger, cfg, load_config, load_model_weight

load_config(cfg, "/content/nanodet_kitti.yml")
nlog = Logger(0, use_tensorboard=False)
CKPT = "/content/nanodet_workspace/model_best/nanodet_model_best.pth"
if not os.path.exists(CKPT):
    CKPT = "/content/nanodet_workspace/model_best/model_best.ckpt"
print("using weights:", CKPT)

class NanoPredictor:
    def __init__(self, device, conf=0.05):
        model = build_model(cfg.model)
        ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
        load_model_weight(model, ckpt, nlog)
        self.model, self.device, self.conf = model.to(device).eval(), device, conf
        self.pipeline = Pipeline(cfg.data.val.pipeline, cfg.data.val.keep_ratio)
    def __call__(self, img):
        h, w = img.shape[:2]
        meta = dict(img_info={"id": 0, "file_name": None, "height": h, "width": w}, raw_img=img, img=img)
        meta = self.pipeline(None, meta, cfg.data.val.input_size)
        meta["img"] = torch.from_numpy(meta["img"].transpose(2, 0, 1)).to(self.device)
        meta = naive_collate([meta])
        meta["img"] = stack_batch_img(meta["img"], divisible=32)
        with torch.no_grad(), contextlib.redirect_stdout(io.StringIO()):
            res = self.model.inference(meta)
        b, s, l = [], [], []
        for c, lst in res[0].items():
            for x1, y1, x2, y2, sc in lst:
                if sc >= self.conf:
                    b.append([x1, y1, x2, y2]); s.append(sc); l.append(int(c))
        return np.array(b, dtype=np.float32).reshape(-1, 4), np.array(s, dtype=np.float32), np.array(l, dtype=np.int64)

cuda = torch.cuda.is_available()
dev = "cuda:0" if cuda else "cpu"
ev = evaluate_model(NanoPredictor(dev, conf=0.05), val_imgs, names)   # NanoDet's own NMS keeps scores >= 0.05
gpu = measure_latency(NanoPredictor("cuda:0", conf=0.25), val_imgs, True) if cuda else None
torch.set_num_threads(CPU_THREADS)
cpu = measure_latency(NanoPredictor("cpu", conf=0.25), val_imgs, False)

pred_model = NanoPredictor("cpu").model
deploy_sd = {k: v for k, v in pred_model.state_dict().items() if not k.startswith("aux_head")}   # aux head is training-only
torch.save(deploy_sd, WTS / "nanodet_deploy.pth")
out = {"tag": "nanodet", "model": "NanoDet-Plus-m", "input_size": INPUT_SIZE, **ev,
       "params_m": sum(p.numel() for n, p in pred_model.named_parameters() if not n.startswith("aux_head")) / 1e6,
       "size_mb": os.path.getsize(WTS / "nanodet_deploy.pth") / 1e6, "onnx_mb": None,
       "gpu_ms": gpu and gpu["mean_ms"], "gpu_fps": gpu and gpu["fps"],
       "cpu_ms": cpu["mean_ms"], "cpu_fps": cpu["fps"], "cpu_p95_ms": cpu["p95_ms"], "cpu_threads": CPU_THREADS,
       "epochs": EPOCHS}
save_result(RES, "nanodet", out)
print(json.dumps(out, indent=2))

## 5. Sample detection images

In [ ]:
import matplotlib.pyplot as plt
pred = NanoPredictor(dev, conf=0.35)
picks = random.Random(3).sample(val_imgs, 3)      # same three images as the main notebook's side-by-side figure
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
for ax, p in zip(axes, picks):
    im = cv2.imread(p)
    b, s, l = pred(im)
    for (x1, y1, x2, y2), sc, c in zip(b, s, l):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (232, 116, 44), 2)
        cv2.putText(im, f"{names.get(int(c), c)} {sc:.2f}", (int(x1), max(12, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    ax.imshow(im[..., ::-1]); ax.axis("off")
axes[0].set_title("NanoDet-Plus-m", loc="left", fontsize=14)
plt.tight_layout(); plt.savefig(FIG / "nanodet_detections.png", dpi=130); plt.show()
print("Result file saved to:", RES / "result_nanodet.json", "- copy it next to the other result files if you run on a different Google account.")